In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('cybercrime_regulation_change_2020_2024.csv')

In [4]:
df.head()

,country_code,country_name,score_2020,score_2024,score_change,percent_change,region,income_level
0,AFG,Afghanistan,0.000000,0.104714,0.104714,inf,Asia,Low Income
1,ALB,Albania,0.932751,1.000000,0.067249,7.21,Europe,Upper Middle Income
2,DZA,Algeria,0.621538,0.922571,0.301033,48.43,Africa,Upper Middle Income
3,AND,Andorra,0.348979,0.815143,0.466164,133.58,Other,Not Classified
4,AGO,Angola,0.347746,0.702000,0.354254,101.87,Africa,Lower Middle Income


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_code    194 non-null    object 
 1   country_name    194 non-null    object 
 2   score_2020      194 non-null    float64
 3   score_2024      194 non-null    float64
 4   score_change    194 non-null    float64
 5   percent_change  190 non-null    float64
 6   region          194 non-null    object 
 7   income_level    194 non-null    object 
dtypes: float64(4), object(4)
memory usage: 12.3+ KB


In [6]:
df.describe()

,score_2020,score_2024,score_change,percent_change
count,194.000000,194.000000,194.000000,190.0000
mean,0.603831,0.741275,0.137444,inf
std,0.380604,0.293498,0.212881,NaN
min,0.000000,0.000000,-0.455238,-62.9800
25%,0.284291,0.552286,0.000000,0.0000
50%,0.646560,0.831071,0.067249,9.7350
75%,1.000000,1.000000,0.234541,95.6825
max,1.000000,1.000000,0.831714,inf


In [7]:
df.isnull().sum()

country_code      0
country_name      0
score_2020        0
score_2024        0
score_change      0
percent_change    4
region            0
income_level      0
dtype: int64

In [8]:
# convertion
df['country_code'] = df['country_code'].astype('string')
df['country_name'] = df['country_name'].astype('string')
df['region'] = df['region'].astype('category')
df['income_level'] = df['income_level'].astype('category')
numeric_cols = ['score_2020','score_2024','score_change','percent_change']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')
# handle infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)     

In [9]:
# Rename columns
df.rename(columns={
    'country_code': 'ISO Country Code',
    'country_name': 'Country',
    'region': 'Geographic Region',
    'income_level': 'World Bank Income Group',
    'score_2020': 'Cyber Regulation Score(2020)',
    'score_2024': 'Cyber Regulation Score(2024)',
    'score_change': 'Absolute Regulatory Progress(2020-2024)',
    'percent_change': 'Relative Regulatory Progress(%)'
}, inplace=True)    

In [10]:
## Create of columns
# 1.Regulatory Maturity Level
def maturity_level(score):
    if score >= 0.80:
        return 'Advanced'
    elif score >= 0.50:
        return 'Developing'
    else:
        return 'Emerging'
df['Regulatory Maturity Level(2024)'] = df['Cyber Regulation Score(2024)'].apply(maturity_level)

 # 2.Regulatory Momentum
def momentum(progress):
    if progress < 0:
        return 'Regression'
    elif progress < 0.05:
        return 'Low Progress'
    elif progress < 0.20:
        return 'Moderate Progress'
    else:
        return 'High Progress'
df['Regulatory Momentum(2020-2024)'] = df['Absolute Regulatory Progress(2020-2024)'].apply(momentum)

 # 3.Structural Gap Indicator
global_median_2024 = df['Cyber Regulation Score(2024)'].median()
df['Structural Gap vs Global Median'] = (df['Cyber Regulation Score(2024)'] - global_median_2024)

 # 4.Strategic Priority Level
def strategic_priority(row):
    if row['Regulatory Maturity Level(2024)']=='Emerging' and \
       row['Regulatory Momentum(2020-2024)'] in ['Low Progress', 'Regression']:
        return 'Critical Priority'
    elif row['Regulatory Maturity Level(2024)'] == 'Emerging':
        return 'High Priority'
    elif row['Regulatory Maturity Level(2024)'] == 'Developing' and \
         row['Regulatory Momentum(2020-2024)'] == 'Low Progress':
        return 'Monitor Closely'
    else:
        return 'Stable / Advanced'
df['Strategic Priority Level'] = df.apply(strategic_priority, axis=1)        

In [11]:
# Columns to drop
columns_to_drop = [
    'ISO Country Code',
    'Cyber Regulation Score(2020)',
    'Relative Regulatory Progress(%)'
]
df.drop(columns=columns_to_drop, inplace=True)

In [13]:
df.to_excel("cybercrime_regulation_change_2020_2024.xlsx",index=False,sheet_name="cybercrime_regulation")

In [16]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [17]:
engine.connect()

In [18]:
df.to_sql(
    name="cybercrime_regulation_change_2020_2024",
    con=engine,
    if_exists="replace",
    index=False
)

194